In [ ]:
# Analyze pymatgen usage of dependency APIs and generate Sankey diagram

In [ ]:
import os
from pathlib import Path

PMG_REPO_PATH = os.getenv("PMG_REPO_PATH")
PMG_COMMIT: str = "ab34799d8ab5dee80756489cf2ca28a97de78121"
!cd "{PMG_REPO_PATH}" && git fetch && git checkout ab34799d8ab5dee80756489cf2ca28a97de78121

MIN_API_USAGE: int = (
    15  # dependency/subpackage with API usage fewer than this would be filtered
)

# Install api-analyzer
api_analyzer_path = Path().resolve() / "api-analyzer"
!uv pip install "{api_analyzer_path}"

# Top-level subpackages of pymatgen
subpacks = [
    "alchemy",
    "analysis",
    "core",
    "electronic_structure",
    "entries",
    "io",
    "phonon",
    "symmetry",
    "transformations",
]

HEAD is now at ab34799d8a Bump rexml from 3.3.9 to 3.4.2 in /docs (#4499)
Using Python 3.14.3 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Resolved 1 package in 0.78ms                                         
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/fig_scr
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/fig_scr
      Built api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/fig_scr
Prepared 1 package in 293ms                                              
Uninstalled 1 package in 0.62ms
Installed 1 package in 1ms (from file:///Users/yang/develope
 ~ api-analyzer==0.1.0 (from file:///Users/yang/developer/pymatgen-2-paper/fig_scripts/pmg-api-usage-dependent-dependency/api-analyzer)


In [ ]:
# Get dependencies (required and optional) of pymatgen
from pathlib import Path
import tomllib
from packaging.requirements import Requirement


def get_all_dependencies(pyproject_path) -> dict[str, list[str] | dict[str, list[str]]]:
    """
    Load dependencies from pyproject.toml and strip versions/markers.
    """

    def _strip_versions(reqs: list[str]) -> list[str]:
        """Return only the package names (no version/markers/extras)."""
        names = []
        for r in reqs:
            names.append(Requirement(r).name)

        return names

    data = tomllib.loads(Path(pyproject_path).read_text(encoding="utf-8"))
    project = data.get("project", {})

    required = project.get("dependencies", [])
    optional = project.get("optional-dependencies", {})

    return {
        "required": _strip_versions(required),
        "optional": {k: _strip_versions(v) for k, v in optional.items()},
    }


deps = get_all_dependencies(f"{PMG_REPO_PATH}/pyproject.toml")
print("Required:", deps["required"])
print("Optional groups:", list(deps["optional"]))

Required: ['bibtexparser', 'joblib', 'matplotlib', 'monty', 'networkx', 'numpy', 'orjson', 'palettable', 'pandas', 'plotly', 'requests', 'ruamel.yaml', 'scipy', 'scipy', 'spglib', 'sympy', 'tabulate', 'tqdm', 'uncertainties']
Optional groups: ['abinit', 'ase', 'electronic_structure', 'matcalc', 'mlp', 'numba', 'numpy-v1', 'optional', 'prototypes', 'symmetry', 'tblite', 'vis', 'zeopp']


In [ ]:
# Analyze each required dependencies' API usage by each pmg subpackage

from pathlib import Path

from api_analyzer import analyze_paths

results: dict[str, dict[str, dict[str, dict[str, int]]]] = {}

for dep in deps["required"]:
    results[dep] = {}
    for pack in subpacks:
        path = Path(PMG_REPO_PATH) / "src" / "pymatgen" / pack
        aliases, usage = analyze_paths(path, package=dep)

        results[dep][pack] = {
            "aliases": aliases,
            "usage": usage,
        }

In [ ]:
!uv pip install plotly nbformat kaleido

import plotly
import plotly.graph_objects as go

# Reuse same color palette
PMG_COLORS = {
    "core": "#3B9C9C",
    "analysis": "#6C5B7B",
    "io": "#355C7D",
    "entries": "#99B898",
    "symmetry": "#FFD3B6",
    "ext": "#E84A5F",
    "electronic_structure": "#F67280",
    "transformations": "#C06C84",
    "phonon": "#A8E6CF",
    "optimization": "#FFAAA5",
    "util": "#FF8B94",
    "command_line": "#0384fc",
    "alchemy": "#6203fc",
}


def hex_to_rgba(hex_color: str, alpha: float = 0.5) -> str:
    """Convert hex color like '#F67280' to rgba(246,114,128,0.5)."""
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i : i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


links: list[tuple[str, str, int]] = []  # (pmg_package, dependency, usage)
package_totals: dict[str, int] = {}
dep_totals: dict[str, int] = {}

for dep, package in results.items():
    for pkg, mod_info in package.items():
        usage = mod_info.get("usage", {}) or {}
        # NOTE: This total usage is per PMG subpackage
        total_usage = sum(usage.values())
        if total_usage > 0:
            links.append((f"pymatgen.{pkg}", dep, total_usage))
            package_totals[pkg] = package_totals.get(pkg, 0) + total_usage
            dep_totals[dep] = dep_totals.get(dep, 0) + total_usage

for pkg, count in package_totals.items():
    if count < MIN_API_USAGE:
        print(f"pymatgen.{pkg=} would be filtered: {count=}, {MIN_API_USAGE=}")

for dep, count in dep_totals.items():
    if count < MIN_API_USAGE:
        print(f"{dep=} would be filtered: {count=}, {MIN_API_USAGE=}")

# Filter out weak dependencies and modules
sorted_deps = [
    dep
    for dep, total in sorted(dep_totals.items(), key=lambda x: -x[1])
    if total >= MIN_API_USAGE
]
sorted_subpackages = [
    f"pymatgen.{m}"
    for m, total in sorted(package_totals.items(), key=lambda x: -x[1])
    if total >= MIN_API_USAGE
]

# Keep only links where both dependency and subpackage survived
valid_deps = set(sorted_deps)
valid_subpackages = set(sorted_subpackages)
links = [
    link for link in links if link[1] in valid_deps and link[0] in valid_subpackages
]


# Labels: dependencies (left) → pymatgen subpackages (right)
def strip_pmg_prefix(name: str) -> str:
    return name.removeprefix("pymatgen.")


labels = [strip_pmg_prefix(lbl) for lbl in sorted_subpackages] + sorted_deps
label_to_idx = {lbl: i for i, lbl in enumerate(labels)}

sources = [label_to_idx[dep] for _, dep, _ in links]
targets = [label_to_idx[strip_pmg_prefix(module)] for module, _, _ in links]
values = [v for _, _, v in links]

# Node colors
node_colors = []
for label in labels:
    if label in [strip_pmg_prefix(x) for x in sorted_subpackages]:
        node_colors.append(PMG_COLORS[label])
    else:
        node_colors.append("#8FB9A8")

# Link colors: match pymatgen subpackage color
link_colors = []
for pkg, _, _ in links:
    sub = pkg.removeprefix("pymatgen.")
    color = PMG_COLORS[sub]
    link_colors.append(hex_to_rgba(color, 0.5))

# Sankey plot
fig = go.Figure(
    go.Sankey(
        arrangement="snap",
        node={
            "label": labels,
            "color": node_colors,
            "line": {"color": "rgba(0,0,0,0.1)", "width": 0.5},
        },
        link={
            "source": sources,
            "target": targets,
            "value": values,
            "color": link_colors,
        },
    )
)

fig.update_layout(
    font={"size": 20},
    height=800,
    width=600,
    margin={"l": 5, "r": 5, "t": 20, "b": 20},
)

fig.show()

# Fix ratio: https://github.com/plotly/Kaleido/issues/378
plotly.io.defaults.default_width = None
plotly.io.defaults.default_height = None

fig.write_image(
    Path.cwd().parents[1] / "paper" / "figs" / "pmg-dependency-usage.pdf",
    format="pdf",
)

Using Python 3.14.3 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Audited 3 packages in 1ms
pymatgen.pkg='alchemy' would be filtered: count=3, MIN_API_USAGE=15
dep='bibtexparser' would be filtered: count=2, MIN_API_USAGE=15
dep='joblib' would be filtered: count=4, MIN_API_USAGE=15
dep='requests' would be filtered: count=3, MIN_API_USAGE=15
dep='ruamel.yaml' would be filtered: count=11, MIN_API_USAGE=15
dep='spglib' would be filtered: count=11, MIN_API_USAGE=15
dep='tabulate' would be filtered: count=7, MIN_API_USAGE=15
dep='tqdm' would be filtered: count=7, MIN_API_USAGE=15
